<a href="https://colab.research.google.com/github/9080965846/EDA-project/blob/main/EDA%20project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set random seed for reproducibility
np.random.seed(42)

n_rows = 10000

# 1. Basic Columns
transaction_id = [f"TXN{100000 + i}" for i in range(n_rows)]
user_ids = [f"USR{np.random.randint(1, 501):03d}" for _ in range(n_rows)]

# 2. Timestamps over last 12 months
end_date = datetime.now()
start_date = end_date - timedelta(days=365)
time_deltas = [np.random.uniform(0, 365) for _ in range(n_rows)]
timestamps = [start_date + timedelta(days=d) for d in time_deltas]

# Helper variables from timestamps for pattern mapping
hours = [t.hour for t in timestamps]
days = [t.day for t in timestamps]
day_of_weeks = [t.weekday() for t in timestamps]  # 5, 6 are weekend
months = [t.month for t in timestamps]             # e.g., Oct (10), Nov (11) as festival months

# 3. Categories and Types
categories = ['Food', 'Groceries', 'Travel', 'Shopping', 'Bills', 'Entertainment', 'Education', 'Health', 'Peer-to-Peer']
apps = ['GPay', 'PhonePe', 'Paytm', 'BHIM']
banks = ['SBI', 'HDFC', 'ICICI', 'Axis', 'PNB']
statuses = ['Success', 'Failed']
city_tiers = ['Tier 1', 'Tier 2', 'Tier 3']
age_groups = ['18-24', '25-34', '35-50', '50+']
devices = ['Android', 'iOS']

# 4. Generate patterns for Merchant Categories and Transaction Types based on temporal features
merchant_category = []
transaction_type = []

for i in range(n_rows):
    hr = hours[i]
    dy = days[i]

    # Realistic distribution weights based on time of day
    if 20 <= hr or hr <= 2:
        # Night: More Food and Entertainment
        weights = [0.35, 0.10, 0.05, 0.10, 0.02, 0.18, 0.01, 0.04, 0.15]
    elif dy <= 5:
        # Month start: Higher chance of Bills and Education
        weights = [0.10, 0.15, 0.05, 0.10, 0.30, 0.05, 0.15, 0.05, 0.05]
    else:
        # Default weights
        weights = [0.15, 0.20, 0.10, 0.15, 0.05, 0.10, 0.05, 0.05, 0.15]

    cat = np.random.choice(categories, p=weights)
    merchant_category.append(cat)

    # P2P vs P2M determination
    if cat in ['Peer-to-Peer']:
        transaction_type.append('P2P')
    else:
        transaction_type.append(np.random.choice(['P2M', 'P2P'], p=[0.85, 0.15]))

# 5. Generate Amounts with Skewed Distribution + Pattern Modifiers
# Base distribution using a log-normal distribution to represent highly skewed amounts (e.g., small tea/grocery payments vs rare large payments)
base_amounts = np.random.lognormal(mean=4.5, sigma=1.2, size=n_rows) # Median around ~90 INR, skew extends to thousands

amounts = []
for i in range(n_rows):
    amt = base_amounts[i]

    # Higher spending on weekends (day_of_week 5 or 6)
    if day_of_weeks[i] >= 5:
        amt *= np.random.uniform(1.2, 1.8)

    # Higher spending in festival months (e.g., October, November, December)
    if months[i] in [10, 11, 12]:
         amt *= np.random.uniform(1.3, 2.0)

    # Specific category adjustments
    cat = merchant_category[i]
    if cat == 'Bills':
        amt += np.random.uniform(200, 1500)
    elif cat == 'Education':
        amt += np.random.uniform(1000, 5000)
    elif cat == 'Food':
        # Keep food transactions within realistic bounds
        amt = min(amt, np.random.uniform(50, 400))

    # Ensure realistic non-zero floor and round
    amt = max(1.0, round(amt, 2))
    amounts.append(amt)

# 6. Fill remaining fields
payment_app = np.random.choice(apps, size=n_rows, p=[0.38, 0.35, 0.20, 0.07])
bank = np.random.choice(banks, size=n_rows, p=[0.30, 0.25, 0.20, 0.15, 0.10])
status = np.random.choice(statuses, size=n_rows, p=[0.95, 0.05]) # 5% failure rate
city_tier = np.random.choice(city_tiers, size=n_rows, p=[0.50, 0.30, 0.20])
age_group = np.random.choice(age_groups, size=n_rows, p=[0.35, 0.40, 0.18, 0.07])
device_type = np.random.choice(devices, size=n_rows, p=[0.75, 0.25])

# 7. Construct DataFrame
df = pd.DataFrame({
    'transaction_id': transaction_id,
    'user_id': user_ids,
    'timestamp': timestamps,
    'amount': amounts,
    'merchant_category': merchant_category,
    'transaction_type': transaction_type,
    'payment_app': payment_app,
    'bank': bank,
    'status': status,
    'city_tier': city_tier,
    'age_group': age_group,
    'device_type': device_type
})

# Sort by timestamp
df = df.sort_values(by='timestamp').reset_index(drop=True)

# Save to CSV
df.to_csv('upi_data.csv', index=False)

# Display summary
print("Dataset shape:", df.shape)
display(df.head())